In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from matplotlib import pyplot
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from nltk import word_tokenize
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, classification_report


In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import wandb
from wandb.integration.keras import WandbCallback, WandbEvalCallback, WandbMetricsLogger


I0000 00:00:1776157320.198273  890643 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [4]:
EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 0.001
DECAY_STEPS = 100
DECAY_RATE = 0.9

In [5]:
def preprocess_pandas(data, columns):
    df_ = pd.DataFrame(columns=columns)
    data['Sentence'] = data['Sentence'].str.lower()
    data['Sentence'] = data['Sentence'].replace('[a-zA-Z0-9-_.]+@[a-zA-Z0-9-_.]+', '', regex=True)                      # remove emails
    data['Sentence'] = data['Sentence'].replace('((25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)(\.|$)){4}', '', regex=True)    # remove IP address
    data['Sentence'] = data['Sentence'].str.replace('[^\w\s]','')                                                       # remove special characters
    data['Sentence'] = data['Sentence'].replace('\d', '', regex=True)                                                   # remove numbers
    for index, row in data.iterrows():
        word_tokens = word_tokenize(row['Sentence'])
        filtered_sent = [w for w in word_tokens if not w in stopwords.words('english')]
        df_.loc[len(df_)] = {
            "index": row['index'],
            "Class": row['Class'],
            "Sentence": " ".join(filtered_sent)
        }
    return data

In [12]:
# If this is the primary file that is executed (ie not an import of another file)
if __name__ == "__main__":
    # get data, pre-process and split
    data = pd.read_csv("amazon_cells_labelled.txt", delimiter='\t', header=None)
    data.columns = ['Sentence', 'Class']
    data['index'] = data.index                                          # add new column index
    columns = ['index', 'Class', 'Sentence']
    data = preprocess_pandas(data, columns)# pre-process
    print(type(data['Sentence'].values.astype('U')), type(data['Class'].values.astype('int32')))
    training_data, validation_data, training_labels, validation_labels = train_test_split( # split the data into training, validation, and test splits
        data['Sentence'].values.astype('U'),
        data['Class'].values.astype('int32'),
        test_size=0.15,
        random_state=0,
        shuffle=True
    )

    # vectorize data using TFIDF and transform for PyTorch for scalability
    word_vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1,2), max_features=50000, max_df=0.5, use_idf=True, norm='l2')
    training_data = word_vectorizer.fit_transform(training_data)        # transform texts to sparse matrix
    training_data = training_data.todense()                             # convert to dense matrix for Pytorch
    vocab_size = len(word_vectorizer.vocabulary_)
    validation_data = word_vectorizer.transform(validation_data)
    validation_data = validation_data.todense()
    train_x_tensor = torch.from_numpy(np.array(training_data)).type(torch.FloatTensor)
    train_y_tensor = torch.from_numpy(np.array(training_labels)).long()
    validation_x_tensor = torch.from_numpy(np.array(validation_data)).type(torch.FloatTensor)
    validation_y_tensor = torch.from_numpy(np.array(validation_labels)).long()
    

<class 'numpy.ndarray'> <class 'numpy.ndarray'>


In [6]:
def train_model(model, name, project_name, train_x, train_y, test_x, test_y, epochs=10, batch_size=32, lr=0.001, decay_steps=100, decay_rate=0.9):

    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=lr,
        decay_steps=decay_steps,
        decay_rate=decay_rate)

    optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',   # surveille la loss sur validation
        patience=10,           # attend 3 epochs sans amélioration
        restore_best_weights=True
    )

    best_model_name = "best_model" + name +".keras"
    checkpoint = tf.keras.callbacks.ModelCheckpoint(
        filepath=best_model_name,          # nom du fichier
        monitor="val_loss",       # on surveille la validation
        save_best_only=True,      # garde seulement le meilleur
        mode="min",               # plus petite loss = mieux
        verbose=1
    )
    
    model.compile(
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy'], 
        optimizer=optimizer
    )
    
    wandb.init(
        project=project_name,
        name=name,
        config={
            "epochs": epochs,
            "batch_size": batch_size,
            "early_stopping": early_stop
        }
    )

    config = wandb.config

    model.fit(
        train_x, train_y,
        epochs=config.epochs,
        batch_size=config.batch_size,
        validation_split=0.15,
        callbacks=[WandbMetricsLogger(), early_stop, checkpoint]
    )
    
    model_name = "LinneaLab1"  + name + ".keras"
    model.save(model_name)
    accuracy = model.evaluate(test_x, test_y)

    print("Accuracy: ", accuracy[1], "Loss: ", accuracy[0])

    wandb.log({
        "test_accuracy": accuracy[1],
        "test_loss": accuracy[0]
    })
    wandb.finish()

In [65]:
model_ann = models.Sequential([
    
    layers.Dense(100, activation='relu', input_shape=(6896,)),
    layers.Dropout(0.2),
    layers.Dense(2, activation='softmax'),
    
])

train_model(model_ann, "ann1", "LinneaLabb1Small",train_x_tensor, train_y_tensor, validation_x_tensor, validation_y_tensor, epochs=100, decay_steps=10)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100


/usr/local/lib/python3.11/dist-packages/keras/src/backend/tensorflow/nn.py:1214: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(
I0000 00:00:1776022853.945265  676058 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_38741__.13


 1/23 ━━━━━━━━━━━━━━━━━━━━ 34s 2s/step - accuracy: 0.5000 - loss: 0.6941

I0000 00:00:1776022854.907158  676061 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_38741__.13


23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 68ms/step - accuracy: 0.5983 - loss: 0.6882 - val_accuracy: 0.7500 - val_loss: 0.6792
Epoch 2/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9695 - loss: 0.6400 - val_accuracy: 0.7891 - val_loss: 0.6582
Epoch 3/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9903 - loss: 0.5676 - val_accuracy: 0.8125 - val_loss: 0.6270
Epoch 4/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9972 - loss: 0.4864 - val_accuracy: 0.8359 - val_loss: 0.5974
Epoch 5/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9986 - loss: 0.4134 - val_accuracy: 0.8203 - val_loss: 0.5728
Epoch 6/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.3615 - val_accuracy: 0.8203 - val_loss: 0.5549
Epoch 7/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.3204 - val_accuracy: 0.8359 - val_loss: 0.5393
Epoch 8/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.2943 - val_accuracy: 0.8281 - val_loss: 0.

epoch/accuracy,▁▇██████████████████████████████████████
epoch/epoch,▁▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇█
epoch/learning_rate,█▆▄▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▇▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▃██████████████████████████████████████
epoch/val_loss,█▇▆▅▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/accuracy,1
epoch/epoch,99
epoch/learning_rate,0.0
epoch/loss,0.21104
epoch/val_accuracy,0.83594


In [56]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

if __name__ == "__main__":
    # get data, pre-process and split
    data_a = pd.read_csv("amazon_cells_labelled.txt", delimiter='\t', header=None)
    data_a.columns = ['Sentence', 'Class']
    data_a['index'] = data.index                                          # add new column index
    columns_a = ['index', 'Class', 'Sentence']
    data_a = preprocess_pandas(data_a, columns_a)                             # pre-process
    training_data_a, validation_data_a, training_labels_a, validation_labels_a = train_test_split( # split the data into training, validation, and test splits
        data_a['Sentence'].values.astype('U'),
        data_a['Class'].values.astype('int32'),
        test_size=0.15,
        random_state=0,
        shuffle=True
    )

tokenizer = Tokenizer()
tokenizer.fit_on_texts(training_data_a)  # your raw sentences list
tokenizer.fit_on_texts(validation_data_a)  # your raw sentences list
sequences_train = tokenizer.texts_to_sequences(training_data_a)
sequences_val = tokenizer.texts_to_sequences(validation_data_a)
#vocab_size = len(tokenizer.word_index) + 1

maxlen = max(len(seq) for seq in sequences)
maxlen = maxlen+10
sequences_pad_train = pad_sequences(sequences_train, maxlen=maxlen)
sequences_pad_val = pad_sequences(sequences_val, maxlen=maxlen)

train_x = torch.from_numpy(np.array(sequences_pad_train)).type(torch.FloatTensor)
train_y = torch.from_numpy(np.array(training_labels_a)).long()
test_x = torch.from_numpy(np.array(sequences_pad_val)).type(torch.FloatTensor)
test_y = torch.from_numpy(np.array(validation_labels_a)).long()


In [66]:
model_lstm = models.Sequential([

    layers.Embedding(vocab_size, 128),
    layers.LSTM(128),
    layers.Dense(2, activation="softmax"),
    
])
train_model(model_lstm, "lstm1", "LinneaLabb1Small",train_x, train_y, test_x, test_y, epochs=100, decay_steps=10)

Epoch 1/100


/usr/local/lib/python3.11/dist-packages/keras/src/backend/tensorflow/nn.py:1214: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.5983 - loss: 0.6834 - val_accuracy: 0.5938 - val_loss: 0.6740
Epoch 2/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7562 - loss: 0.5719 - val_accuracy: 0.7734 - val_loss: 0.5184
Epoch 3/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8643 - loss: 0.3847 - val_accuracy: 0.7891 - val_loss: 0.4516
Epoch 4/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9377 - loss: 0.2555 - val_accuracy: 0.8281 - val_loss: 0.3960
Epoch 5/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9626 - loss: 0.1514 - val_accuracy: 0.8203 - val_loss: 0.3968
Epoch 6/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9848 - loss: 0.1051 - val_accuracy: 0.8672 - val_loss: 0.3612
Epoch 7/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9917 - loss: 0.0788 - val_accuracy: 0.8594 - val_loss: 0.3699
Epoch 8/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9917 - loss: 0.0626 - val_accuracy: 0.8672 - val_los

epoch/accuracy,▁▆██████████████████████████████████████
epoch/epoch,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇█
epoch/learning_rate,█▄▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▂▅▅▇▇██▇███████████████████████████████
epoch/val_loss,▄▁▂▂▄▇▇█████████████████████████████████
epoch/accuracy,0.99584
epoch/epoch,99
epoch/learning_rate,0.0
epoch/loss,0.02679
epoch/val_accuracy,0.86719


In [67]:
model_bilstm = models.Sequential([

    layers.Embedding(vocab_size, 128),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True)),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
    layers.Dense(2, activation="softmax"),
    
])

train_model(model_bilstm, "bilstm", "LinneaLabb1Small",train_x, train_y, test_x, test_y, epochs=100, decay_steps=10)

Epoch 1/100


/usr/local/lib/python3.11/dist-packages/keras/src/backend/tensorflow/nn.py:1214: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


23/23 ━━━━━━━━━━━━━━━━━━━━ 5s 55ms/step - accuracy: 0.5623 - loss: 0.6886 - val_accuracy: 0.5859 - val_loss: 0.6755
Epoch 2/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.7867 - loss: 0.5370 - val_accuracy: 0.7812 - val_loss: 0.4348
Epoch 3/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.9183 - loss: 0.2114 - val_accuracy: 0.8281 - val_loss: 0.4416
Epoch 4/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.9543 - loss: 0.1316 - val_accuracy: 0.8203 - val_loss: 0.4544
Epoch 5/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.9792 - loss: 0.0743 - val_accuracy: 0.8203 - val_loss: 0.4628
Epoch 6/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.9889 - loss: 0.0491 - val_accuracy: 0.8281 - val_loss: 0.5762
Epoch 7/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.9945 - loss: 0.0311 - val_accuracy: 0.8203 - val_loss: 0.6502
Epoch 8/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.9958 - loss: 0.0213 - val_accuracy: 0.8516 - val_l

epoch/accuracy,▁▄▇█████████████████████████████████████
epoch/epoch,▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▇▇▇▇▇▇███
epoch/learning_rate,█▅▄▃▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▃▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▅▅█▅▆▅▅▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
epoch/val_loss,▁▁▂▆▆▇▇█████████████████████████████████
epoch/accuracy,1
epoch/epoch,99
epoch/learning_rate,0.0
epoch/loss,0.0024
epoch/val_accuracy,0.8125


In [5]:
if __name__ == "__main__":
    # get data, pre-process and split
    data = pd.read_csv("amazon_cells_labelled_LARGE_25K.txt", delimiter='\t', header=None)
    data.columns = ['Sentence', 'Class']
    data['index'] = data.index                                          # add new column index
    columns = ['index', 'Class', 'Sentence']
    data = preprocess_pandas(data, columns)                             # pre-process
    training_data, validation_data, training_labels, validation_labels = train_test_split( # split the data into training, validation, and test splits
        data['Sentence'].values.astype('U'),
        data['Class'].values.astype('int32'),
        test_size=0.15,
        random_state=0,
        shuffle=True
    )

    # vectorize data using TFIDF and transform for PyTorch for scalability
    word_vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1,2), max_features=50000, max_df=0.5, use_idf=True, norm='l2')
    training_data = word_vectorizer.fit_transform(training_data)        # transform texts to sparse matrix
    training_data = training_data.todense()                             # convert to dense matrix for Pytorch
    vocab_size = len(word_vectorizer.vocabulary_)
    validation_data = word_vectorizer.transform(validation_data)
    validation_data = validation_data.todense()
    train_x_tensor = torch.from_numpy(np.array(training_data)).type(torch.FloatTensor)
    train_y_tensor = torch.from_numpy(np.array(training_labels)).long()
    validation_x_tensor = torch.from_numpy(np.array(validation_data)).type(torch.FloatTensor)
    validation_y_tensor = torch.from_numpy(np.array(validation_labels)).long()

In [7]:
print(train_x_tensor.size())

torch.Size([21250, 50000])


In [23]:
model_ann = models.Sequential([
    
    layers.Dense(100, activation='relu', input_shape=(50000,)),
    layers.Dropout(0.2),
    layers.Dense(2, activation='softmax'),
    
])

train_model(model_ann, "ann3", "LinneaLabb125k", train_x_tensor, train_y_tensor, validation_x_tensor, validation_y_tensor, epochs=100, decay_steps=10, lr=0.0001)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100


/usr/local/lib/python3.11/dist-packages/keras/src/backend/tensorflow/nn.py:1214: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(
I0000 00:00:1776028435.722015  789683 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_500577__.13


564/565 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6046 - loss: 0.6810

I0000 00:00:1776028439.443079  789680 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_500577__.13


565/565 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6046 - loss: 0.6810
Epoch 1: val_loss improved from None to 0.67244, saving model to best_modelann3.keras

Epoch 1: finished saving model to best_modelann3.keras
565/565 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.6064 - loss: 0.6757 - val_accuracy: 0.6010 - val_loss: 0.6724
Epoch 2/100
564/565 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6043 - loss: 0.6707
Epoch 2: val_loss improved from 0.67244 to 0.67238, saving model to best_modelann3.keras

Epoch 2: finished saving model to best_modelann3.keras
565/565 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.6061 - loss: 0.6703 - val_accuracy: 0.6010 - val_loss: 0.6724
Epoch 3/100
564/565 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6059 - loss: 0.6703
Epoch 3: val_loss improved from 0.67238 to 0.67238, saving model to best_modelann3.keras

Epoch 3: finished saving model to best_modelann3.keras
565/565 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.6061 - loss: 0.6703 - val_acc

epoch/accuracy,█▂▂▂▁▁▁▁▁▂▁▁▂
epoch/epoch,▁▂▂▃▃▄▅▅▆▆▇▇█
epoch/learning_rate,█▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_loss,█▁▁▁▁▁▁▁▁▁▁▁▁
test_accuracy,▁
test_loss,▁
epoch/accuracy,0.60613
epoch/epoch,12
epoch/learning_rate,0.0


In [13]:
if __name__ == "__main__":
    # get data, pre-process and split
    data_a = pd.read_csv("amazon_cells_labelled_LARGE_25K.txt", delimiter='\t', header=None)
    data_a.columns = ['Sentence', 'Class']
    data_a['index'] = data.index                                          # add new column index
    columns_a = ['index', 'Class', 'Sentence']
    data_a = preprocess_pandas(data_a, columns_a)                             # pre-process
    training_data_a, validation_data_a, training_labels_a, validation_labels_a = train_test_split( # split the data into training, validation, and test splits
        data_a['Sentence'].values.astype('U'),
        data_a['Class'].values.astype('int32'),
        test_size=0.15,
        random_state=0,
        shuffle=True
    )

In [15]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(training_data_a)  # your raw sentences list
tokenizer.fit_on_texts(validation_data_a)  # your raw sentences list
sequences_train = tokenizer.texts_to_sequences(training_data_a)
sequences_val = tokenizer.texts_to_sequences(validation_data_a)
#vocab_size = len(tokenizer.word_index) + 1

maxlen = max(len(seq) for seq in sequences_train)
maxlen = maxlen+10
sequences_pad_train = pad_sequences(sequences_train, maxlen=maxlen)
sequences_pad_val = pad_sequences(sequences_val, maxlen=maxlen)

train_x = torch.from_numpy(np.array(sequences_pad_train)).type(torch.FloatTensor)
train_y = torch.from_numpy(np.array(training_labels_a)).long()
test_x = torch.from_numpy(np.array(sequences_pad_val)).type(torch.FloatTensor)
test_y = torch.from_numpy(np.array(validation_labels_a)).long()

In [25]:
model_lstm = models.Sequential([

    layers.Embedding(vocab_size, 128),
    layers.LSTM(128),
    layers.Dense(2, activation="softmax"),
    
])
train_model(model_lstm, "lstm3", "LinneaLabb125k",train_x, train_y, test_x, test_y, epochs=100, decay_steps=10, lr=0.0001)

Epoch 1/100


/usr/local/lib/python3.11/dist-packages/keras/src/backend/tensorflow/nn.py:1214: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


565/565 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6023 - loss: 0.6713
Epoch 1: val_loss improved from None to 0.66050, saving model to best_modellstm3.keras

Epoch 1: finished saving model to best_modellstm3.keras
565/565 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.6057 - loss: 0.6637 - val_accuracy: 0.6010 - val_loss: 0.6605
Epoch 2/100
561/565 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6055 - loss: 0.6582
Epoch 2: val_loss improved from 0.66050 to 0.66045, saving model to best_modellstm3.keras

Epoch 2: finished saving model to best_modellstm3.keras
565/565 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.6061 - loss: 0.6582 - val_accuracy: 0.6010 - val_loss: 0.6604
Epoch 3/100
565/565 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6015 - loss: 0.6601
Epoch 3: val_loss improved from 0.66045 to 0.66045, saving model to best_modellstm3.keras

Epoch 3: finished saving model to best_modellstm3.keras
565/565 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.6061 - loss: 0.65

epoch/accuracy,▁████████████
epoch/epoch,▁▂▂▃▃▄▅▅▆▆▇▇█
epoch/learning_rate,█▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_loss,█▁▁▁▁▁▁▁▁▁▁▁▁
test_accuracy,▁
test_loss,▁
epoch/accuracy,0.60608
epoch/epoch,12
epoch/learning_rate,0.0


In [26]:
model_bilstm = models.Sequential([

    layers.Embedding(vocab_size, 128),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True)),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
    layers.Dense(2, activation="softmax"),
    
])

#nr3 lr 0.0001
#nr2 lr 0.001


train_model(model_bilstm, "bilstm3", "LinneaLabb125k",train_x, train_y, test_x, test_y, epochs=100, decay_steps=10, lr=0.0001)

Epoch 1/100


/usr/local/lib/python3.11/dist-packages/keras/src/backend/tensorflow/nn.py:1214: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


564/565 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.6051 - loss: 0.6700
Epoch 1: val_loss improved from None to 0.66508, saving model to best_modelbilstm3.keras

Epoch 1: finished saving model to best_modelbilstm3.keras
565/565 ━━━━━━━━━━━━━━━━━━━━ 19s 28ms/step - accuracy: 0.6060 - loss: 0.6656 - val_accuracy: 0.6010 - val_loss: 0.6651
Epoch 2/100
565/565 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.6086 - loss: 0.6614
Epoch 2: val_loss improved from 0.66508 to 0.66503, saving model to best_modelbilstm3.keras

Epoch 2: finished saving model to best_modelbilstm3.keras
565/565 ━━━━━━━━━━━━━━━━━━━━ 15s 26ms/step - accuracy: 0.6061 - loss: 0.6625 - val_accuracy: 0.6010 - val_loss: 0.6650
Epoch 3/100
565/565 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.6020 - loss: 0.6642
Epoch 3: val_loss did not improve from 0.66503
565/565 ━━━━━━━━━━━━━━━━━━━━ 14s 25ms/step - accuracy: 0.6061 - loss: 0.6624 - val_accuracy: 0.6010 - val_loss: 0.6650
Epoch 4/100
564/565 ━━━━━━━━━━━━━━━━━━━━ 0s 2

epoch/accuracy,▁███████████
epoch/epoch,▁▂▂▃▄▄▅▅▆▇▇█
epoch/learning_rate,█▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_loss,█▁▁▁▁▁▁▁▁▁▁▁
test_accuracy,▁
test_loss,▁
epoch/accuracy,0.60608
epoch/epoch,11
epoch/learning_rate,0.0


In [6]:
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("rogate16/amazon-reviews-2018-full-dataset")

print("Path to dataset files:", path)



/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /root/.cache/kagglehub/datasets/rogate16/amazon-reviews-2018-full-dataset/versions/1


In [7]:
sentences = []
labels = []
for data in pd.read_csv(path + "/amazon_reviews.csv", chunksize=50000):
    data = data.dropna(subset=['reviewText', 'rating'])
    sentences.extend(data['reviewText'].values.astype('U').tolist())
    labels.extend(data['rating'].values.astype('int32').tolist())
    print("done")

done
done
done
done
done
done
done
done
done
done
done
done


In [9]:
#data = data.dropna(subset=['reviewText', 'rating'])
print(min(labels))  # should print 0
print(max(labels)) 


1
5


In [8]:
#sentences = data['reviewText'].values.astype('U')
#labels = data['rating'].values.astype('int32')
labels = [l - 1 for l in labels]
print(min(labels))  # should print 0
print(max(labels)) 

0
4


In [9]:
print(min(labels))  # should print 0
print(max(labels)) 

0
4


In [9]:

print(type(sentences))
print(len(sentences))
print(len(labels))

training_data_a, validation_data_a, training_labels_a, validation_labels_a = train_test_split( # split the data into training, validation, and test splits
        sentences,
        labels,
        test_size=0.15,
        random_state=0,
        shuffle=True
    )

print(type(training_data_a))
tokenizer = Tokenizer()
tokenizer.fit_on_texts(training_data_a)  # your raw sentences list
tokenizer.fit_on_texts(validation_data_a)  # your raw sentences list
sequences_train = tokenizer.texts_to_sequences(training_data_a)
sequences_val = tokenizer.texts_to_sequences(validation_data_a)
vocab_size = len(tokenizer.word_index) + 1

maxlen = max(len(seq) for seq in sequences_train)
maxlen = maxlen+10
sequences_pad_train = pad_sequences(sequences_train, maxlen=maxlen)
sequences_pad_val = pad_sequences(sequences_val, maxlen=maxlen)

train_x = torch.from_numpy(np.array(sequences_pad_train)).type(torch.FloatTensor)
train_y = torch.from_numpy(np.array(training_labels_a)).long()
test_x = torch.from_numpy(np.array(sequences_pad_val)).type(torch.FloatTensor)
test_y = torch.from_numpy(np.array(validation_labels_a)).long()

train_dataset = TensorDataset(train_x, train_y)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

val_dataset = TensorDataset(test_x, test_y)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)



<class 'list'>
550731
550731
<class 'list'>


In [11]:
print(train_x.size())
print(train_y[1])

torch.Size([468121, 5175])
tensor(4)


In [10]:
model_ann = models.Sequential([
    
    layers.Dense(100, activation='relu', input_shape=(5175,)),
    layers.Dropout(0.2),
    layers.Dense(5, activation='softmax'),
    
])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1776157409.219373  890643 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6792 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:21:00.0, compute capability: 7.5


In [11]:
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=0.001,
        decay_steps=10,
        decay_rate=0.9)

#optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',   # surveille la loss sur validation
    patience=50,           # attend 3 epochs sans amélioration
    restore_best_weights=True
)

best_model_name = "best_model" + "ann" +".keras"
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=best_model_name,          # nom du fichier
    monitor="val_loss",       # on surveille la validation
    save_best_only=True,      # garde seulement le meilleur
    mode="min",               # plus petite loss = mieux
    verbose=1
)

model_ann.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'], 
    optimizer=optimizer
)

wandb.init(
    project="Task1.3",
    name="ann1gig",
    config={
        "epochs": 100,
        "early_stopping": early_stop, 
        
    }
)

config = wandb.config

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: linneahejsupergroup (linneahejsupergroup-lule-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
model_ann.fit(
    train_loader,
    epochs=config.epochs,
    validation_data=val_loader,
    callbacks=[WandbMetricsLogger(), early_stop, checkpoint]
)

model_name = "LinneaLab1.3"  + "ann" + ".keras"
model_ann.save(model_name)

Epoch 1/100


I0000 00:00:1776157414.056675  890643 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
/usr/local/lib/python3.11/dist-packages/keras/src/backend/tensorflow/nn.py:1214: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(
I0000 00:00:1776157414.940871  890960 service.cc:153] XLA service 0x7f559c031850 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776157414.940916  890960 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 2080 Ti, Compute Capability 7.5 (Driver: 13.0.0; Runtime: 13.0.0; Toolkit: 12.5.0; DNN: 9.19.0)
I0000 00:00:1776157415.280539  890960 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1776157415.870443  890960 cuda_dnn.cc:461]

    7/14629 ━━━━━━━━━━━━━━━━━━━━ 5:39 23ms/step - accuracy: 0.1386 - loss: 410.2043

I0000 00:00:1776157418.649866  890960 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


14627/14629 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.4566 - loss: 145.3443

I0000 00:00:1776157661.070751  890959 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_988__.13


14629/14629 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.4566 - loss: 145.3374
Epoch 1: val_loss improved from None to 36.51476, saving model to best_modelann.keras

Epoch 1: finished saving model to best_modelann.keras
14629/14629 ━━━━━━━━━━━━━━━━━━━━ 261s 17ms/step - accuracy: 0.4963 - loss: 94.9824 - val_accuracy: 0.6146 - val_loss: 36.5148
Epoch 2/100
14618/14629 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5446 - loss: 26.5603
Epoch 2: val_loss improved from 36.51476 to 7.36356, saving model to best_modelann.keras

Epoch 2: finished saving model to best_modelann.keras
14629/14629 ━━━━━━━━━━━━━━━━━━━━ 72s 5ms/step - accuracy: 0.5788 - loss: 18.0439 - val_accuracy: 0.6585 - val_loss: 7.3636
Epoch 3/100
 5453/14629 ━━━━━━━━━━━━━━━━━━━━ 39s 4ms/step - accuracy: 0.6632 - loss: 4.9614

In [11]:
model_lstm = models.Sequential([

    layers.Embedding(vocab_size, 128),
    layers.LSTM(128),
    layers.Dense(2, activation="softmax"),
    
])

I0000 00:00:1776114962.633434  884450 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6794 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:21:00.0, compute capability: 7.5


In [12]:
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=0.001,
        decay_steps=10,
        decay_rate=0.9)

optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',   # surveille la loss sur validation
    patience=50,           # attend 3 epochs sans amélioration
    restore_best_weights=True
)

best_model_name = "best_model" + "ann" +".keras"
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=best_model_name,          # nom du fichier
    monitor="val_loss",       # on surveille la validation
    save_best_only=True,      # garde seulement le meilleur
    mode="min",               # plus petite loss = mieux
    verbose=1
)

model_lstm.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'], 
    optimizer=optimizer
)

wandb.init(
    project=" ",
    name="lstm",
    config={
        "epochs": 100,
        "early_stopping": early_stop
    }
)

config = wandb.config

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: linneahejsupergroup (linneahejsupergroup-lule-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
model_lstm.fit(
    train_loader,
    epochs=config.epochs,
    validation_data=val_loader,
    callbacks=[WandbMetricsLogger(), early_stop, checkpoint]
)

model_name = "LinneaLab1"  + "lstm" + ".keras"
model_lstm.save(model_name)

Epoch 1/100


I0000 00:00:1776114988.124426  884450 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
/usr/local/lib/python3.11/dist-packages/keras/src/backend/tensorflow/nn.py:1214: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(
I0000 00:00:1776114994.399332  884924 cuda_dnn.cc:461] Loaded cuDNN version 91900


13138/14629 ━━━━━━━━━━━━━━━━━━━━ 5:06 206ms/step - accuracy: 0.0623 - loss: nan

KeyboardInterrupt: 